In [1]:
from datetime import datetime

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


In [2]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

EXTRACT_TRAIN_PARQUET = f"{OUTPUT_PREFIX}/extract_train_v2.parquet"
EXTRACT_TEST_PARQUET = f"{OUTPUT_PREFIX}/extract_test_v2.parquet"
SUBMISSION_CSV = f"{DATA_PREFIX}/sample_submission.csv"


In [11]:
BANDS_S2 = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
# BANDS_S2_DIST = [f"{b}_DIST" for b in BANDS_S2]
BANDS_S1 = ["VV", "VH"]
BANDS_CHM = [
# "CHM_META",
"CHM_ETH"
]
BANDS_TERRAIN = ["elevation", "slope", "aspect", "tpi", "tri", "hillshade"]
# BANDS_S1_DIST = [f"{b}_DIST" for b in BANDS_S1]

INDICES = [
    dict(name="NDVI", band1="NIR", band2="RED"),
    dict(name="NDMI", band1="NIR", band2="SWIR1"),
    dict(name="NBR", band1="NIR", band2="SWIR2"),
    dict(name="NBR2", band1="SWIR1", band2="SWIR2"),
    dict(name="NDWI", band1="GREEN", band2="NIR"),
    dict(name="MNDWI", band1="GREEN", band2="SWIR1"),
    dict(name="MNDWI2", band1="GREEN", band2="SWIR2"),
    dict(name="RVI", band1="VV", band2="VH"),
]

INDICES_BANDS = [indi["name"] for indi in INDICES]
# INDICES_BANDS_DIST = [f"{b}_DIST" for b in INDICES_BANDS]

PREDICTORS = [
    *BANDS_S2,
    *BANDS_S1,
    *INDICES_BANDS,
    *BANDS_CHM,
    *BANDS_TERRAIN
    # *BANDS_S2_DIST,
    # *BANDS_S1_DIST,
    # *INDICES_BANDS_DIST,
]

LABEL = "biomass"


In [12]:
# load parquet extracted
train_df = gpd.read_parquet(EXTRACT_TRAIN_PARQUET)
train_df

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,VH,CHM,CHM_META,CHM_ETH,elevation,slope,aspect,tri,tpi,hillshade
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,303.0,3.0,11,11,777,19.786716,67.750977,14.730920,0.125,150.0
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,465.0,8.0,15,15,900,24.269398,337.833649,21.166010,-3.750,209.0
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,346.0,5.0,13,13,800,22.094799,356.760315,16.062378,-0.250,198.0
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,356.0,0.0,13,13,753,22.413956,211.328705,16.401220,-0.375,185.0
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,383.0,7.0,13,13,856,11.787092,222.137589,9.219544,0.375,189.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,304.0,0.0,16,16,827,2.943096,315.000000,3.464102,0.750,185.0
5148500,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261),353.0,593.0,527.0,3129.0,...,278.0,2.0,15,15,826,0.000000,NaN,1.732051,-0.125,181.0
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,375.0,5.0,12,12,820,1.316193,71.565048,1.414214,0.000,179.0
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,439.0,4.0,12,12,821,2.426940,120.963760,2.236068,0.125,177.0


In [13]:
# Generating indices
def generate_indices(table, band_suffix=""):
    for index_dict in INDICES:
        name = f"{index_dict['name']}{band_suffix}"
        band1 = f"{index_dict['band1']}{band_suffix}"
        band2 = f"{index_dict['band2']}{band_suffix}"
        table[name] = (
            ((table[band1] / 1e4) - (table[band2] / 1e4))
            / ((table[band1] / 1e4) + (table[band2] / 1e4))
            * 1e4
        )


generate_indices(train_df)
# generate_indices(train_df, "_DIST")

train_df

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,tpi,hillshade,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,0.125,150.0,6082.148499,953.303206,3286.348212,2408.500590,-6246.409192,-5628.252788,-3724.653148,5634.005764
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,-3.750,209.0,7891.138575,2368.660244,5231.699127,3268.015171,-7260.273973,-5907.540735,-3271.028037,5745.654163
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,-0.250,198.0,5931.465869,908.752328,3172.925568,2331.396817,-6097.829074,-5493.494129,-3626.570916,5230.875258
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,-0.375,185.0,3526.570048,-428.312890,1646.684610,2060.465116,-4745.972739,-5071.200232,-3362.035225,4729.829756
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,0.375,189.0,7243.589744,2396.313364,4677.174901,2568.768515,-6497.444634,-4857.428714,-2614.942529,4161.585366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,0.750,185.0,4213.197970,1301.259283,3425.393172,2223.230490,-4730.639731,-3654.333502,-1557.653405,6197.623515
5148500,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261),353.0,593.0,527.0,3129.0,...,-0.125,181.0,7117.067834,1279.740447,3726.694451,2569.498571,-6813.541107,-6062.416999,-4137.419674,6678.614098
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,0.000,179.0,6973.029589,1990.381058,4546.678636,2810.650888,-7035.479632,-5866.617809,-3659.359191,4710.860367
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,0.125,177.0,7311.254561,2233.240777,4431.445952,2439.644219,-7161.936561,-5867.098865,-4000.000000,5334.750266


In [14]:
train_df_filter = train_df[(train_df["BLUE"] > 0) & (train_df["VV"] > 0)]
train_df_filter

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,tpi,hillshade,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,0.125,150.0,6082.148499,953.303206,3286.348212,2408.500590,-6246.409192,-5628.252788,-3724.653148,5634.005764
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,-3.750,209.0,7891.138575,2368.660244,5231.699127,3268.015171,-7260.273973,-5907.540735,-3271.028037,5745.654163
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,-0.250,198.0,5931.465869,908.752328,3172.925568,2331.396817,-6097.829074,-5493.494129,-3626.570916,5230.875258
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,-0.375,185.0,3526.570048,-428.312890,1646.684610,2060.465116,-4745.972739,-5071.200232,-3362.035225,4729.829756
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,0.375,189.0,7243.589744,2396.313364,4677.174901,2568.768515,-6497.444634,-4857.428714,-2614.942529,4161.585366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,0.750,185.0,4213.197970,1301.259283,3425.393172,2223.230490,-4730.639731,-3654.333502,-1557.653405,6197.623515
5148500,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261),353.0,593.0,527.0,3129.0,...,-0.125,181.0,7117.067834,1279.740447,3726.694451,2569.498571,-6813.541107,-6062.416999,-4137.419674,6678.614098
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,0.000,179.0,6973.029589,1990.381058,4546.678636,2810.650888,-7035.479632,-5866.617809,-3659.359191,4710.860367
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,0.125,177.0,7311.254561,2233.240777,4431.445952,2439.644219,-7161.936561,-5867.098865,-4000.000000,5334.750266


In [15]:
# split train and test data
train, test = train_test_split(train_df_filter, test_size=0.3)

In [ ]:
MODEL_NAME = f"XGB_v1_{datetime.now().timestamp()}"
model = XGBRegressor(n_estimators=500)
model.fit(train[PREDICTORS], train[LABEL])

In [ ]:
total = sum(model.feature_importances_)
print("Feature importance")
pd.Series(dict(zip(model.feature_names_in_, model.feature_importances_ / total * 100)))

Feature importance


BLUE       0.640749
GREEN      1.419845
RED       16.779270
NIR        0.432734
SWIR1      1.852753
SWIR2      5.826331
VV         0.430790
VH         0.179202
NDVI      10.118524
NDMI      14.323627
NBR        5.711577
NBR2       0.732167
NDWI      11.180109
MNDWI      0.860601
MNDWI2     0.205374
RVI        0.202072
CHM       29.104275
dtype: float32

In [ ]:
test_apply = model.predict(test[PREDICTORS])
r2 = np.corrcoef(test[LABEL], test_apply)[0, 1] ** 2
mae = mean_absolute_error(test[LABEL], test_apply)
print(f"R^2={r2}", f"MAE={mae}")

R^2=0.7313246300250252 MAE=18.681184578304823


In [ ]:
test_df = gpd.read_parquet(EXTRACT_TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,SWIR1,SWIR2,VV,VH,CHM
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,1293.0,561.0,2425.0,548.0,17.0
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,1256.0,539.0,1591.0,403.0,20.0
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,1171.0,486.0,1360.0,447.0,10.0
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,1318.0,559.0,1893.0,504.0,22.0
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,1180.0,431.0,1943.0,350.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),289.0,583.0,396.0,4034.0,2086.0,1174.0,1288.0,360.0,1.0
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),686.0,1111.0,1201.0,4128.0,2865.0,2012.0,1127.0,282.0,0.0
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),286.0,574.0,375.0,3594.0,2122.0,1218.0,1332.0,356.0,11.0
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),210.0,405.0,271.0,3095.0,1383.0,667.0,1321.0,381.0,15.0


In [ ]:
generate_indices(test_df)
# generate_indices(test_df, "_DIST")
test_df

,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,...,VH,CHM,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,...,548.0,17.0,9136.363636,4451.834370,7144.311530,3948.220065,-8274.552360,-6052.141527,-2764.505119,6313.488059
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,...,403.0,20.0,8949.024544,4112.959925,6963.380282,3994.428969,-8089.516371,-5959.339263,-2578.763127,5957.873621
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,...,447.0,10.0,9287.605295,5477.018154,7836.634765,4133.977067,-8478.210745,-5602.931379,-1911.764706,5052.573326
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,...,504.0,22.0,9014.167651,4192.553426,7042.328042,4043.686734,-8146.478873,-6004.857316,-2590.090090,5794.743429
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,...,350.0,16.0,9428.704134,5599.477904,8131.772865,4649.286158,-8762.054272,-6208.791209,-2192.362093,6947.230702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),289.0,583.0,396.0,4034.0,...,360.0,1.0,8212.189616,3183.006536,5491.551459,2797.546012,-7474.550574,-5631.322593,-3363.688105,5631.067961
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),686.0,1111.0,1201.0,4128.0,...,282.0,0.0,5492.587728,1806.091806,3446.254072,1749.026041,-5758.732583,-4411.468813,-2885.046430,5997.161107
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),286.0,574.0,375.0,3594.0,...,356.0,11.0,8110.355253,2575.227432,4937.655860,2706.586826,-7245.681382,-5741.839763,-3593.750000,5781.990521
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),210.0,405.0,271.0,3095.0,...,381.0,15.0,8389.780154,3823.135328,6454.013822,3492.682927,-7685.714286,-5469.798658,-2444.029851,5522.914219


In [ ]:
submission_df = pd.read_csv(SUBMISSION_CSV)
submission_df

,Id,Expected
0,028631_3040064_2251003,0.0
1,028631_3041214_2252063,0.0
2,028631_3041184_2251693,0.0
3,028631_3039864_2250783,0.0
4,028631_3040224_2251483,0.0
...,...,...
45014,061751_3374644_2043173,0.0
45015,061751_3374594_2043203,0.0
45016,061751_3374624_2043173,0.0
45017,061751_3374614_2043223,0.0


In [ ]:
submission_df["Expected"] = model.predict(test_df[PREDICTORS])
submission_df


,Id,Expected
0,028631_3040064_2251003,273.703644
1,028631_3041214_2252063,290.669067
2,028631_3041184_2251693,249.534805
3,028631_3039864_2250783,276.789307
4,028631_3040224_2251483,298.540497
...,...,...
45014,061751_3374644_2043173,52.763626
45015,061751_3374594_2043203,24.716759
45016,061751_3374624_2043173,99.357841
45017,061751_3374614_2043223,164.722565


In [ ]:
RESULT_CSV = f"{OUTPUT_PREFIX}/results_{MODEL_NAME}.csv"
submission_df.to_csv(RESULT_CSV, index=False)

MODEL_NAME

'XGBRF_v1_1787140937.225513'